In [ ]:
from pathlib import Path

BOOKS_DIR = Path("/kaggle/input/datasets/mishasavinov/ceramic-net-dataset-books/books")
META_CSV_PATH = BOOKS_DIR / "meta.csv"
OUTPUT_DIR = Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
!pip install -q timm pytorch-metric-learning onnx onnxruntime albumentations faiss-cpu

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from pytorch_metric_learning import losses

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
df = pd.read_csv(META_CSV_PATH)

def find_image_path(row):
    rel = str(row.get("image_path", "")).strip().replace("\\", "/")
    book = str(row.get("book", "")).strip()
    mark_id = str(row.get("mark_id", "")).strip()
    filename = str(row.get("filename", f"{mark_id}.png")).strip()
    
    candidates = [
        BOOKS_DIR / rel if rel else None,
        BOOKS_DIR / book / "marks" / filename,
        BOOKS_DIR / book / "marks" / f"{mark_id}.png",
        BOOKS_DIR / book / filename,
        BOOKS_DIR / filename
    ]
    for p in candidates:
        if p is not None and p.is_file():
            return str(p)
    return None

df["full_path"] = df.apply(find_image_path, axis=1)
valid_df = df[df["full_path"].notna()].copy().reset_index(drop=True)

valid_df["mark_id_clean"] = valid_df["mark_id"].astype(str).str.strip()
mark_to_id = {mark: idx for idx, mark in enumerate(valid_df["mark_id_clean"].unique())}
valid_df["class_id"] = valid_df["mark_id_clean"].map(mark_to_id)
NUM_CLASSES = len(mark_to_id)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

paths = valid_df["full_path"].tolist()

with ThreadPoolExecutor(max_workers=4) as executor:
    cached_images = list(executor.map(read_and_letterbox_book, paths))

class FastCachedDataset(Dataset):
    def __init__(self, pil_images, class_ids):
        self.images = pil_images
        self.labels = class_ids

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx: int):
        img_pil = self.images[idx]
        class_id = self.labels[idx]

        view_anchor = clean_transform(img_pil)
        view_fragment = heavy_crop_transform(img_pil)
        return view_anchor, view_fragment, torch.tensor(class_id, dtype=torch.long)

dataset = FastCachedDataset(cached_images, valid_df["class_id"].values)
train_loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
)

In [ ]:
class DINOv2MetricEncoder(nn.Module):
    def __init__(self, out_dim: int = 384, unfreeze_blocks: int = 1):
        super().__init__()
        self.backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14")
        for p in self.backbone.parameters():
            p.requires_grad = False
            
        if unfreeze_blocks > 0:
            for p in self.backbone.blocks[-unfreeze_blocks:].parameters():
                p.requires_grad = True
            for p in self.backbone.norm.parameters():
                p.requires_grad = True
                
        self.projector = nn.Sequential(
            nn.Linear(384, 512),
            nn.BatchNorm1d(512),
            nn.PReLU(),
            nn.Linear(512, out_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(x)
        proj = self.projector(feats)
        return F.normalize(proj, p=2, dim=-1)

model = DINOv2MetricEncoder(out_dim=384, unfreeze_blocks=1).to(device)

In [ ]:
criterion = losses.SupConLoss(temperature=0.07).to(device)

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

EPOCHS = 15
scaler = torch.amp.GradScaler("cuda")

for epoch in range(1, EPOCHS + 1):
    model.train()
    
    for anchors, fragments, labels in train_loader:
        anchors = anchors.to(device, non_blocking=True)
        fragments = fragments.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        combined_images = torch.cat([anchors, fragments], dim=0)
        combined_labels = torch.cat([labels, labels], dim=0)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            embeddings = model(combined_images)
            loss = criterion(embeddings, combined_labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        scaler.step(optimizer)
        scaler.update()

    scheduler.step()

torch.save(model.state_dict(), OUTPUT_DIR / "stamp_encoder_best.pt")

In [ ]:
model.eval()
all_embs = []
all_ids = []

with torch.no_grad():
    for idx in range(len(valid_df)):
        row = valid_df.iloc[idx]
        img_pil = read_and_letterbox_book(row["full_path"])
        tensor = clean_transform(img_pil).unsqueeze(0).to(device)
        emb = model(tensor).cpu().numpy().squeeze(0)
        
        all_embs.append(emb)
        all_ids.append(str(row["mark_id_clean"]))

all_embs = np.vstack(all_embs).astype(np.float32)
all_ids = np.array(all_ids)

np.savez_compressed(
    OUTPUT_DIR / "reference_index.npz",
    embeddings=all_embs,
    mark_ids=all_ids
)

In [ ]:
model.eval()
dummy = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE, device=device)
onnx_out = OUTPUT_DIR / "stamp_encoder.onnx"

torch.onnx.export(
    model,
    (dummy,),
    str(onnx_out),
    input_names=["input"],
    output_names=["embedding"],
    dynamo=False,
    dynamic_axes={"input": {0: "batch"}, "embedding": {0: "batch"}},
    opset_version=14,
)